# Chapitre 2 — Leçon 5 : Premier Modèle Complet

## Objectifs d'apprentissage

À la fin de cette leçon, vous serez capable de :
- **Exécuter** le workflow ML complet de A à Z sur un dataset réel
- **Comparer** votre modèle à une baseline pour valider son utilité
- **Aller au-delà de `.score()`** avec des métriques plus informatives
- **Sauvegarder** votre pipeline entraîné pour une utilisation ultérieure

---

## 🎯 Accroche : Le test de réalité

Votre modèle affiche **87% d'accuracy**. Impressionnant ?

Pas si vite. Et si je vous disais que prédire toujours la classe majoritaire (sans aucun modèle) donne **85% d'accuracy** ?

Soudain, votre "excellent" modèle n'est que **2% meilleur qu'un algorithme stupide**.

**Question :** Comment savoir si votre modèle a vraiment appris quelque chose d'utile ?

*(Réponse attendue : En le comparant à une baseline — un modèle "naïf" qui sert de référence)*

Dans cette leçon, nous allons construire un modèle complet **et** le valider correctement.

---

## Configuration et imports

In [ ]:
# Imports standards
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn : split, preprocessing, modèles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Modèles
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier  # ← Pour la baseline !

# Métriques
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# Sauvegarde
import joblib

print("✅ Tous les imports chargés")

---

## 5.1 Chargement et exploration du dataset

Nous allons utiliser un dataset de **prédiction de churn** (départ de clients) — un cas d'usage classique en entreprise.

In [ ]:
# Créer un dataset réaliste de churn télécom
np.random.seed(42)
n = 1000

# Générer les features
data = pd.DataFrame({
    'anciennete_mois': np.random.randint(1, 72, n),
    'montant_mensuel': np.random.uniform(20, 100, n).round(2),
    'nb_appels_support': np.random.poisson(2, n),
    'minutes_utilisees': np.random.randint(100, 1000, n),
    'type_contrat': np.random.choice(['Mensuel', 'Annuel', 'Bi-annuel'], n, p=[0.5, 0.35, 0.15]),
    'paiement': np.random.choice(['Carte', 'Prélèvement', 'Chèque'], n, p=[0.4, 0.45, 0.15]),
})

# Ajouter des valeurs manquantes réalistes
data.loc[np.random.choice(n, 30), 'nb_appels_support'] = np.nan
data.loc[np.random.choice(n, 20), 'paiement'] = np.nan

# Générer le churn (target) basé sur des règles réalistes
churn_proba = (
    0.1 +  # Base
    0.3 * (data['type_contrat'] == 'Mensuel').astype(float) +  # Contrat court = plus de churn
    0.2 * (data['nb_appels_support'].fillna(0) > 3).astype(float) +  # Beaucoup d'appels = frustration
    -0.15 * (data['anciennete_mois'] > 24).astype(float)  # Clients fidèles partent moins
)
data['churn'] = (np.random.random(n) < churn_proba).astype(int)

print("📊 Dataset Churn Télécom")
print("=" * 50)
print(f"Taille : {len(data)} clients")
print(f"\nAperçu :")
data.head()

In [ ]:
# Exploration rapide
print("📋 Informations sur le dataset")
print("=" * 50)
print(f"\n1. Types de données :")
print(data.dtypes)
print(f"\n2. Valeurs manquantes :")
print(data.isnull().sum())
print(f"\n3. Distribution de la target (churn) :")
print(data['churn'].value_counts(normalize=True).map('{:.1%}'.format))

**Question :** En regardant la distribution du churn, quel problème potentiel identifiez-vous ?

*(Réponse attendue : Déséquilibre des classes — le churn est moins fréquent, ce qui peut biaiser les métriques)*

---

## 5.2 Préparation des données

Suivons le workflow appris :
1. Séparer X et y
2. Identifier colonnes numériques et catégorielles
3. Train/test split avec stratification

In [ ]:
# Séparer features et target
X = data.drop('churn', axis=1)
y = data['churn']

# Identifier les types de colonnes
colonnes_numeriques = ['anciennete_mois', 'montant_mensuel', 'nb_appels_support', 'minutes_utilisees']
colonnes_categorielles = ['type_contrat', 'paiement']

print(f"Features (X) : {X.shape[1]} colonnes")
print(f"  - Numériques : {colonnes_numeriques}")
print(f"  - Catégorielles : {colonnes_categorielles}")
print(f"\nTarget (y) : churn (0 = reste, 1 = part)")

In [ ]:
# Train/test split avec stratification (important pour classes déséquilibrées !)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,        # ← Préserve les proportions de churn
    random_state=42
)

print(f"Train : {len(X_train)} clients ({y_train.mean():.1%} de churn)")
print(f"Test  : {len(X_test)} clients ({y_test.mean():.1%} de churn)")
print(f"\n✅ Proportions préservées grâce à la stratification")

---

## 5.3 Construire le pipeline

Appliquons le pattern appris dans la leçon précédente.

In [ ]:
# Pipeline pour colonnes numériques
pipeline_num = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline pour colonnes catégorielles
pipeline_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combiner avec ColumnTransformer
preprocesseur = ColumnTransformer([
    ('num', pipeline_num, colonnes_numeriques),
    ('cat', pipeline_cat, colonnes_categorielles)
])

# Pipeline complet
pipeline = Pipeline([
    ('preprocesseur', preprocesseur),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42
    ))
])

print("✅ Pipeline créé")
print(pipeline)

---

## 5.4 Entraîner et évaluer

### Étape 1 : Entraîner le modèle

In [ ]:
# Entraîner
pipeline.fit(X_train, y_train)
print("✅ Pipeline entraîné !")

# Score de base
score_train = pipeline.score(X_train, y_train)
score_test = pipeline.score(X_test, y_test)

print(f"\n📊 Performance (Accuracy) :")
print(f"  - Train : {score_train:.2%}")
print(f"  - Test  : {score_test:.2%}")

### 🎯 Étape 2 : Comparer avec une BASELINE

**C'est ici que beaucoup de débutants s'arrêtent — mais pas vous !**

Un modèle avec 85% d'accuracy, c'est bien ? Ça dépend ! Si prédire toujours "0" donne aussi 85%, votre modèle n'a rien appris.

```
┌─────────────────────────────────────────────────────────────────┐
│                    L'IMPORTANCE DE LA BASELINE                  │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   SANS BASELINE :                                               │
│   "Mon modèle a 87% d'accuracy !" → Est-ce bon ? 🤷             │
│                                                                 │
│   AVEC BASELINE :                                               │
│   "Baseline stupide : 72%"                                      │
│   "Mon modèle : 87%"                                            │
│   "Amélioration : +15 points" → Oui, c'est bon ! ✅             │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
# BASELINE 1 : Prédire toujours la classe majoritaire
baseline_majoritaire = DummyClassifier(strategy='most_frequent')
baseline_majoritaire.fit(X_train, y_train)
score_baseline_maj = baseline_majoritaire.score(X_test, y_test)

# BASELINE 2 : Prédire aléatoirement selon la distribution
baseline_stratified = DummyClassifier(strategy='stratified', random_state=42)
baseline_stratified.fit(X_train, y_train)
score_baseline_strat = baseline_stratified.score(X_test, y_test)

print("📊 Comparaison avec les Baselines")
print("=" * 50)
print(f"\n{'Modèle':<35} {'Accuracy':>10}")
print("-" * 50)
print(f"{'Baseline (toujours majoritaire)':<35} {score_baseline_maj:>10.2%}")
print(f"{'Baseline (aléatoire stratifié)':<35} {score_baseline_strat:>10.2%}")
print(f"{'Notre RandomForest':<35} {score_test:>10.2%}")
print("-" * 50)

amelioration = score_test - score_baseline_maj
print(f"\n🎯 Amélioration vs baseline : +{amelioration:.1%}")

if amelioration > 0.05:
    print("✅ Le modèle apporte une valeur significative !")
elif amelioration > 0:
    print("⚠️ Amélioration modeste — peut-être optimiser ?")
else:
    print("❌ Le modèle ne fait pas mieux que deviner !")

<details>
<summary>🤔 Question Socratique : Pourquoi DummyClassifier est-il si important même s'il semble "stupide" ?</summary>

### 🔑 Réponse

`DummyClassifier` représente le **seuil minimum** qu'un vrai modèle doit battre. Il répond à la question fondamentale :

**"Mon modèle a-t-il vraiment appris quelque chose, ou aurait-on pu deviner ?"**

Exemples concrets :
- Dataset avec 95% de classe 0 → Baseline = 95% (prédire toujours 0)
- Si votre modèle fait 94% → Il est **pire** que ne rien faire !
- Si votre modèle fait 97% → Amélioration réelle de +2 points

**Règle :** Toujours calculer et rapporter la baseline dans vos projets. C'est la marque d'un data scientist professionnel.

</details>

### 🔍 Étape 3 : Aller au-delà de `.score()` (Accuracy)

L'accuracy seule peut être **trompeuse**, surtout avec des classes déséquilibrées. Regardons d'autres métriques.

In [ ]:
# Faire les prédictions
y_pred = pipeline.predict(X_test)

# Matrice de confusion : VISUALISER les erreurs
print("📊 Matrice de Confusion")
print("=" * 50)
print("\n(Montre OÙ le modèle se trompe)\n")

cm = confusion_matrix(y_test, y_pred)
print(f"                    Prédit")
print(f"                  0       1")
print(f"Réel  0        {cm[0,0]:4}    {cm[0,1]:4}  ← Clients fidèles")
print(f"      1        {cm[1,0]:4}    {cm[1,1]:4}  ← Churners")
print(f"")
print(f"Interprétation :")
print(f"  - {cm[0,0]} clients fidèles correctement identifiés ✓")
print(f"  - {cm[0,1]} clients fidèles prédits comme churners (Faux Positifs)")
print(f"  - {cm[1,0]} churners manqués ! (Faux Négatifs) ← Problème business")
print(f"  - {cm[1,1]} churners correctement détectés ✓")

In [ ]:
# Visualiser la matrice
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Reste', 'Churn'],
    cmap='Blues',
    ax=ax
)
ax.set_title('Matrice de Confusion\n(Prédiction de Churn)')
plt.tight_layout()
plt.show()

In [ ]:
# Rapport de classification complet
print("📊 Rapport de Classification Complet")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=['Reste', 'Churn']))

### Comprendre les métriques

```
┌─────────────────────────────────────────────────────────────────┐
│                    MÉTRIQUES DE CLASSIFICATION                  │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   PRECISION : "Parmi ceux que j'ai prédits churners,           │
│                combien le sont vraiment ?"                      │
│                                                                 │
│   RECALL :    "Parmi les vrais churners,                        │
│                combien ai-je détectés ?"                        │
│                                                                 │
│   F1-SCORE :  Moyenne harmonique de precision et recall         │
│               (équilibre les deux)                              │
│                                                                 │
│   ─────────────────────────────────────────────────────────     │
│                                                                 │
│   💼 DANS LE CONTEXTE CHURN :                                   │
│                                                                 │
│   • Precision faible = On dérange des clients fidèles           │
│                        (offres inutiles, coûts marketing)       │
│                                                                 │
│   • Recall faible = On manque des churners !                    │
│                     (perte de revenus, clients perdus)          │
│                                                                 │
│   Le choix dépend du COÛT BUSINESS de chaque type d'erreur.     │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

**Question :** Pour la prédiction de churn, vaut-il mieux optimiser la precision ou le recall ?

*(Réponse attendue : En général, le recall — mieux vaut contacter un client fidèle "par erreur" que de laisser partir un churner sans réagir)*

---

## 5.5 Sauvegarder le modèle

Un modèle non sauvegardé est un modèle perdu ! Voici comment persister votre pipeline pour l'utiliser plus tard.

In [ ]:
# Sauvegarder le pipeline complet
chemin_modele = 'pipeline_churn.pkl'
joblib.dump(pipeline, chemin_modele)

print(f"✅ Pipeline sauvegardé dans : {chemin_modele}")
print(f"\n📦 Ce fichier contient TOUT :")
print(f"   - Le preprocesseur (scaler, encoder)")
print(f"   - Le modèle entraîné (RandomForest)")
print(f"   - Les paramètres appris")

In [ ]:
# Charger et utiliser le modèle sauvegardé
pipeline_charge = joblib.load(chemin_modele)

print("📂 Pipeline rechargé depuis le fichier")
print("\nTest sur de nouvelles données :")

# Simuler un nouveau client
nouveau_client = pd.DataFrame([{
    'anciennete_mois': 3,
    'montant_mensuel': 89.99,
    'nb_appels_support': 5,
    'minutes_utilisees': 250,
    'type_contrat': 'Mensuel',
    'paiement': 'Carte'
}])

# Prédire
prediction = pipeline_charge.predict(nouveau_client)[0]
proba = pipeline_charge.predict_proba(nouveau_client)[0]

print(f"\n👤 Nouveau client : contrat {nouveau_client['type_contrat'].values[0]}, {nouveau_client['nb_appels_support'].values[0]} appels support")
print(f"   → Prédiction : {'🔴 CHURN PROBABLE' if prediction == 1 else '🟢 Client fidèle'}")
print(f"   → Probabilité de churn : {proba[1]:.1%}")

### 📖 Définition

```
┌─────────────────────────────────────────────────────────────────┐
│ 📖 DÉFINITION : Persistance du modèle                           │
│                                                                 │
│ La persistance (ou sérialisation) consiste à sauvegarder        │
│ un modèle entraîné dans un fichier pour le réutiliser sans      │
│ avoir à le ré-entraîner.                                        │
│                                                                 │
│ En Python, deux options principales :                           │
│ • joblib.dump() / joblib.load() — Recommandé pour sklearn       │
│ • pickle.dump() / pickle.load() — Standard Python               │
│                                                                 │
│ Format typique : .pkl ou .joblib                                │
│                                                                 │
│ En production, ce fichier est chargé par le serveur pour        │
│ faire des prédictions en temps réel.                            │
└─────────────────────────────────────────────────────────────────┘
```

<details>
<summary>🤔 Question Socratique : Pourquoi sauvegarder le PIPELINE entier plutôt que juste le modèle ?</summary>

### 🔑 Réponse

Si vous ne sauvegardez que le modèle (`RandomForestClassifier`), vous devrez :

1. Recréer le preprocesseur à la main
2. Espérer qu'il soit identique à celui de l'entraînement
3. Risquer des incohérences (mauvais scaling, encoding différent...)

En sauvegardant le **Pipeline complet** :

1. TOUT est inclus (preprocessing + modèle)
2. Vous pouvez appeler `.predict()` directement sur des données brutes
3. Zéro risque d'incohérence

```python
# ❌ Risqué : seulement le modèle
joblib.dump(pipeline.named_steps['classifier'], 'model_only.pkl')

# ✅ Recommandé : le pipeline complet
joblib.dump(pipeline, 'pipeline_complet.pkl')
```

</details>

In [ ]:
# Nettoyer : supprimer le fichier de test
import os
if os.path.exists(chemin_modele):
    os.remove(chemin_modele)
    print(f"🗑️ Fichier {chemin_modele} supprimé (nettoyage du notebook)")

---

## 📋 Récapitulatif : Le workflow complet

Voici le workflow que vous venez de maîtriser :

In [ ]:
# =====================================================
# WORKFLOW ML COMPLET - RÉCAPITULATIF
# =====================================================

print("""
┌─────────────────────────────────────────────────────────────────┐
│                 WORKFLOW ML COMPLET - CHECKLIST                 │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ✅ ÉTAPE 1 : CHARGER ET EXPLORER                               │
│     - data.head(), data.info(), data.describe()                 │
│     - Identifier types de colonnes et valeurs manquantes        │
│     - Vérifier la distribution de la target                     │
│                                                                 │
│  ✅ ÉTAPE 2 : PRÉPARER LES DONNÉES                              │
│     - Séparer X et y                                            │
│     - Train/test split AVEC stratify pour classification        │
│     - random_state pour reproductibilité                        │
│                                                                 │
│  ✅ ÉTAPE 3 : CONSTRUIRE LE PIPELINE                            │
│     - Pipeline numérique (Imputer + Scaler)                     │
│     - Pipeline catégoriel (Imputer + Encoder)                   │
│     - ColumnTransformer pour combiner                           │
│     - Pipeline final (preprocesseur + modèle)                   │
│                                                                 │
│  ✅ ÉTAPE 4 : ENTRAÎNER                                         │
│     - pipeline.fit(X_train, y_train)                            │
│                                                                 │
│  ✅ ÉTAPE 5 : ÉVALUER                                           │
│     - Calculer la BASELINE (DummyClassifier)                    │
│     - Comparer accuracy : modèle vs baseline                    │
│     - Matrice de confusion pour voir les erreurs                │
│     - Precision, Recall, F1 pour vue complète                   │
│                                                                 │
│  ✅ ÉTAPE 6 : SAUVEGARDER                                       │
│     - joblib.dump(pipeline, 'model.pkl')                        │
│     - En production : joblib.load() + .predict()                │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
""")

---

## 🧠 Réflexion métacognitive

1. **Pourquoi la baseline est-elle maintenant une étape incontournable** dans votre workflow ?

2. **Quelle métrique choisiriez-vous** pour un problème de détection de fraude ? De diagnostic médical ?

3. **Êtes-vous confiant** pour reproduire ce workflow sur un nouveau dataset ?

---

## 📝 Résumé du chapitre complet

| Leçon | Concept clé | Ce que vous savez faire |
|-------|-------------|------------------------|
| 2.1 | Pipeline ML | Décrire les 5 étapes : Data → Preprocess → Train → Evaluate → Deploy |
| 2.2 | Train/Test/Val | Séparer correctement, éviter le data leakage, stratifier |
| 2.3 | Fit/Predict | Utiliser l'API universelle sklearn : `.fit()`, `.predict()`, `.score()` |
| 2.4 | Pipelines | Construire des pipelines avec ColumnTransformer |
| 2.5 | Modèle complet | Baseline, métriques au-delà d'accuracy, sauvegarde |

**Vous êtes prêt pour le Chapitre 3 : Algorithmes ML Essentiels !**

---

## ➡️ Prochaine étape

Dans le **Chapitre 3 : Algorithmes ML Essentiels**, nous allons explorer les algorithmes les plus utilisés :
- Régression linéaire et logistique
- Arbres de décision et Random Forest
- K-Means clustering

**Question de transition :** Vous savez maintenant *comment* entraîner un modèle. Mais comment choisir *quel* algorithme utiliser pour un problème donné ?